In [1]:
import os
import sys
from pathlib import Path

# Thêm thư mục gốc dự án vào sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
from dotenv import load_dotenv

# Tải cấu hình môi trường từ thư mục gốc
load_dotenv(ROOT_DIR / ".env")

# Nạp các phân hệ cốt lõi của RAG Banking
from rag_banking.config import Config
from rag_banking.chunking.breadcrumbs_chunker import BreadcrumbsChunker
from rag_banking.embeddings.embedding_model import EmbeddingModel
from rag_banking.vector_db.cosine_hnsw_store import CosineHNSWVectorStore
from rag_banking.keyword_search.bm25_search import BM25Search
from rag_banking.query_parser.rag_query_parser import BankingQueryParser
from rag_banking.retrieval.hybrid_retriever import HybridRetriever
from rag_banking.reranking.rag_reranker import BankingReranker
from rag_banking.synthesizer.context_synthesizer import ContextSynthesizer
from rag_banking.generator.rag_generator import RAGGenerator, BANKING_SYSTEM_GUARDRAILS
from rag_banking.pipeline import BankingRAGPipeline

print("Đã nạp thành công toàn bộ thư viện và mô-đun của Hệ thống AI Banking RAG!")


Đã nạp thành công toàn bộ thư viện và mô-đun của Hệ thống AI Banking RAG!


D:\Desktop\GitHub\AI Banking Asistant\.venv\Lib\site-packages\langsmith\_internal\_beta_decorator.py:70: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  if asyncio.iscoroutinefunction(func):


## PHA INGESTION & DUAL-INDEXING (LẬP CHỈ MỤC KÉP SONG SONG)

### 1. Phân đoạn bảo toàn Breadcrumbs:
Mỗi tài liệu văn bản quy phạm, biểu phí và hướng dẫn được phân đoạn giữ nguyên đường dẫn phân cấp:
$$\text{Breadcrumbs} = \text{Tên tài liệu} > \text{Chương/Mục} > \text{Điều X} > \text{Khoản Y / Câu hỏi}$$
Nội dung chunk luôn có tiền tố: `[Breadcrumbs: ...]\n[Pháp lý: [Điều X, Khoản Y]]\nNội dung`.

### 2. Chỉ mục kép song song (Dual-Indexing):
* **Sparse Index**: Giải thuật **Okapi BM25** với tham số $k_1 = 1.5, b = 0.75$, tính Document Frequency và Inverse Document Frequency (IDF) chuẩn xác.
* **Dense Vector Store**: Không gian vector **1024 chiều** với khoảng cách **Cosine Distance** và cấu trúc đồ thị **HNSW** (Hierarchical Navigable Small World, $M=16, ef_{construct}=100$).


In [2]:
# Khởi tạo Pipeline hoàn chỉnh
pipeline = BankingRAGPipeline()

# Danh sách tài liệu ngân hàng thực tế đại diện từ knowledge_base
curated_files = [
    "phap_ly__quy_che_ekyc_ngân hàng_han_muc.md",
    "phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md",
    "phap_ly__dieu_kien_dieu_khoan_mo_va_su_dung_tai_khoan_khcn.md",
    "quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac_thuc_nfc.md",
    "faq__ngan_hang_so_vcb_digibank_va_smart_otp.md",
    "ngan_hang_so__quy_chuan_dich_vu_digibank_va_han_muc.md",
    "tai_khoan__bieu_phi_dich_vu_tai_khoan_thanh_toan.md",
    "cho_vay__vay_tieu_dung_co_tai_san_bao_dam.md"
]

all_chunks = []
dataset_dir = Config.RETRIEVAL_DATASET_DIR

for fname in curated_files:
    fpath = dataset_dir / fname
    if fpath.exists():
        text = fpath.read_text(encoding="utf-8", errors="ignore")
        chunks = pipeline.chunker.chunk_text(text, source_name=fname)
        all_chunks.extend(chunks)

print(f"-> Đã trích xuất tổng cộng: {len(all_chunks)} chunks có cấu trúc Breadcrumbs.")

# Lập chỉ mục kép song song:
# 1. Nạp Dense Vector Store (Cosine HNSW 1024-d)
texts = [c["content"] for c in all_chunks]
print(f"-> Đang sinh vector nhúng 1024 chiều (multilingual-e5-large) cho {len(texts)} chunks...")
vectors = pipeline.embedding_model.embed_documents(texts, batch_size=32)
pipeline.vector_store.add_chunks(all_chunks, vectors)
print(f"   [Dense Index] Đã lưu {len(vectors)} vector (1024 chiều) vào Cosine HNSW Vector Store.")

# 2. Lập chỉ mục Sparse Okapi BM25
pipeline.bm25_search.index(all_chunks)
pipeline.indexed_chunks = all_chunks
print(f"   [Sparse Index] Đã lập chỉ mục Okapi BM25 trên {len(all_chunks)} chunks.")

# Preview 1 chunk tiêu biểu
sample_chunk = [c for c in all_chunks if c.get("legal_ref")][0]
print("\n" + "=" * 65)
print("BẢN XEM TRƯỚC (PREVIEW) 1 CHUNK TIÊU BIỂU:")
print("=" * 65)
print(f"Chunk ID: {sample_chunk['chunk_id']}")
print(f"Nguồn: {sample_chunk['source']}")
print(f"Breadcrumbs: {sample_chunk['breadcrumb']}")
print(f"Thẻ pháp lý: {sample_chunk['legal_ref']}")
print(f"Nội dung:\n{sample_chunk['content'][:350]}...")


  KHỞI TẠO HỆ THỐNG AI BANKING RAG ASSISTANT (BANKING DUAL-INDEX)
[1/6] Khởi tạo Breadcrumbs Chunker (Bảo toàn phân cấp & Thẻ pháp lý)...
[2/6] Khởi tạo Dense Embedding Model 1024 chiều (multilingual-e5-large)...
[3/6] Khởi tạo Dense Vector Store (Cosine HNSW 1024 chiều)...
[4/6] Khởi tạo Sparse Keyword Search (Okapi BM25 k1=1.5, b=0.75)...
[5/6] Khởi tạo Query Parser, Hybrid Retriever (RRF k=60) & Reranker...
[6/6] Khởi tạo Context Synthesizer & LLM Serving (DeepSeek-V4-Flash) kèm Banking Guardrails...
[Hệ thống] Toàn bộ 5 phân hệ đã sẵn sàng hoạt động!

-> Đã trích xuất tổng cộng: 151 chunks có cấu trúc Breadcrumbs.
-> Đang sinh vector nhúng 1024 chiều (multilingual-e5-large) cho 151 chunks...


   [Dense Index] Đã lưu 151 vector (1024 chiều) vào Cosine HNSW Vector Store.
   [Sparse Index] Đã lập chỉ mục Okapi BM25 trên 151 chunks.

BẢN XEM TRƯỚC (PREVIEW) 1 CHUNK TIÊU BIỂU:
Chunk ID: phap_ly__quy_che_ekyc_ngân hàng_han_muc_bc_0
Nguồn: phap_ly__quy_che_ekyc_ngân hàng_han_muc.md
Breadcrumbs: QUY CHẾ MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TRỰC TUYẾN EKYC BANKING
Thẻ pháp lý: [Quyết định 2345/QĐ-NHNN]
Nội dung:
[Breadcrumbs: QUY CHẾ MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TRỰC TUYẾN EKYC BANKING]
[Pháp lý: [Quyết định 2345/QĐ-NHNN]]
*Mã văn bản: QC-eKYC-VCB-2026 | Ban hành bởi: Ngân hàng TMCP Ngoại thương Việt Nam*
*Căn cứ pháp lý: Thông tư 16/2020/TT-NHNN, Thông tư 17/2024/TT-NHNN và Quyết định 2345/QĐ-NHNN*...


## BƯỚC 1: QUERY PARSER (CHUẨN HÓA TỪ VIẾT TẮT & TRÍCH XUẤT TỪ KHÓA)

Phân hệ **Query Parser** thực hiện:
1. **Chuẩn hóa từ viết tắt nghiệp vụ**:
   - `eKYC` $\rightarrow$ `định danh điện tử (eKYC)`
   - `TKTT` $\rightarrow$ `tài khoản thanh toán (TKTT)`
   - `Smart OTP` $\rightarrow$ `phương thức xác thực Smart OTP`
   - `VCB Digibank` $\rightarrow$ `ngân hàng số VCB Digibank (Ngân hàng Digibank)`
   - `KHCN` $\rightarrow$ `khách hàng cá nhân (KHCN)`
   - `CCCD` $\rightarrow$ `căn cước công dân gắn chip (CCCD)`
2. **Nhận diện thực thể pháp lý**: Trích xuất `Điều X`, `Khoản Y` nếu có trong câu hỏi.
3. **Trích xuất từ khóa (Keywords)**: Tách các thuật ngữ trọng tâm phục vụ truy vấn BM25.


In [3]:
test_queries = [
    "Hạn mức giao dịch và điều kiện mở TKTT trực tuyến qua eKYC trên app VCB Digibank xác thực Smart OTP là gì?",
    "Theo Điều 1 và Điều 2, khách hàng mở TKTT qua eKYC có quyền và nghĩa vụ gì?",
    "Phí duy trì tài khoản thanh toán và phí sử dụng dịch vụ VCB Digibank mỗi tháng là bao nhiêu?"
]

print("=" * 80)
print("KIỂM THỬ BƯỚC 1: QUERY PARSER TRÊN CÁC CÂU HỎI THỰC TẾ")
print("=" * 80)

parsed_results = []
for idx, q in enumerate(test_queries, start=1):
    parsed = pipeline.step1_query_parser(q)
    parsed_results.append(parsed)
    print(f"\n[CÂU HỎI {idx}]")
    print(f"  * Câu gốc:        {parsed.raw_query}")
    print(f"  * Câu chuẩn hóa:  {parsed.normalized_query}")
    print(f"  * Từ viết tắt:    {parsed.detected_acronyms}")
    print(f"  * Thực thể pháp lý: {parsed.legal_references}")
    print(f"  * Từ khóa BM25:   {parsed.keywords[:8]}")


KIỂM THỬ BƯỚC 1: QUERY PARSER TRÊN CÁC CÂU HỎI THỰC TẾ

[CÂU HỎI 1]
  * Câu gốc:        Hạn mức giao dịch và điều kiện mở TKTT trực tuyến qua eKYC trên app VCB Digibank xác thực Smart OTP là gì?
  * Câu chuẩn hóa:  Hạn mức giao dịch và điều kiện mở tài khoản thanh toán (TKTT) trực tuyến qua định danh điện tử (eKYC) trên app ngân hàng số VCB Digibank (Ngân hàng Digibank) xác thực phương thức xác thực Smart OTP là gì?
  * Từ viết tắt:    [('VCB Digibank', 'ngân hàng số VCB Digibank (Ngân hàng Digibank)'), ('Smart OTP', 'phương thức xác thực Smart OTP'), ('eKYC', 'định danh điện tử (eKYC)'), ('TKTT', 'tài khoản thanh toán (TKTT)')]
  * Thực thể pháp lý: []
  * Từ khóa BM25:   ['ekyc', 'tktt', 'smart otp', 'vcb digibank', 'hạn', 'mức', 'giao', 'dịch']

[CÂU HỎI 2]
  * Câu gốc:        Theo Điều 1 và Điều 2, khách hàng mở TKTT qua eKYC có quyền và nghĩa vụ gì?
  * Câu chuẩn hóa:  Theo Điều 1 và Điều 2, khách hàng mở tài khoản thanh toán (TKTT) qua định danh điện tử (eKYC) có quyền và nghĩa v

## BƯỚC 2: HYBRID RETRIEVAL (TRUY VẤN SONG SONG & RRF k=60)

Truy vấn đồng thời hai luồng:
1. **Luồng Dense**: Embedding câu truy vấn bằng vector 1024-d, tìm kiếm trên Cosine HNSW Vector Store.
2. **Luồng Sparse**: Tìm kiếm khớp từ khóa chính xác bằng Okapi BM25.
3. **Reciprocal Rank Fusion (RRF $k=60$)**:
$$RRF(d) = \frac{0.5}{60 + \text{rank}_{\text{dense}}(d)} + \frac{0.5}{60 + \text{rank}_{\text{bm25}}(d)}$$

Thuật toán kết hợp cả hai thứ hạng và trích xuất đúng **Top-5 ứng viên** tiềm năng nhất.


In [4]:
primary_query = parsed_results[0]
candidates_top5 = pipeline.step2_hybrid_retrieval(primary_query, top_k=5)

# Trình bày kết quả Top-5 dưới dạng DataFrame trực quan
df_top5 = []
for c in candidates_top5:
    chunk = c["chunk"]
    df_top5.append({
        "Top (RRF)": c["candidate_rank"],
        "RRF Score": c["rrf_score"],
        "Dense Rank": c["dense_rank"] if c["dense_rank"] else "-",
        "Dense Score": c["dense_score"],
        "BM25 Rank": c["bm25_rank"] if c["bm25_rank"] else "-",
        "BM25 Score": c["bm25_score"],
        "Nguồn tài liệu": chunk.get("source"),
        "Căn cứ pháp lý": chunk.get("legal_ref", "Quy chế"),
        "Đường dẫn (Breadcrumbs)": chunk.get("breadcrumb", "")[:45] + "..."
    })

df_display_top5 = pd.DataFrame(df_top5)
print("KẾT QUẢ BƯỚC 2: TOP-5 ỨNG VIÊN HỢP NHẤT BẰNG THUẬT TOÁN RRF (k=60)")
display(df_display_top5)


KẾT QUẢ BƯỚC 2: TOP-5 ỨNG VIÊN HỢP NHẤT BẰNG THUẬT TOÁN RRF (k=60)


,Top (RRF),RRF Score,Dense Rank,Dense Score,BM25 Rank,BM25 Score,Nguồn tài liệu,Căn cứ pháp lý,Đường dẫn (Breadcrumbs)
0,1,0.016009,1,0.9258,4,38.7014,phap_ly__quy_che_ekyc_ngân hàng_han_muc.md,"[Điều 12, Khoản 1]",QUY CHẾ MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TR...
1,2,0.016001,3,0.8990,2,39.1558,quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac...,,QUY TRÌNH HƯỚNG DẪN MỞ TÀI KHOẢN THANH TOÁN V...
2,3,0.015388,4,0.8985,6,33.9531,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,,BỘ CÂU HỎI THƯỜNG GẶP (FAQ): NGÂN HÀNG SỐ VIE...
3,4,0.014953,14,0.8868,1,40.2041,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,,BỘ CÂU HỎI THƯỜNG GẶP (FAQ): NGÂN HÀNG SỐ VIE...
4,5,0.014914,2,0.9002,13,26.8506,phap_ly__quy_che_ekyc_ngân hàng_han_muc.md,"[Điều 12, Khoản 2; Quyết định 2345/QĐ-NHNN]",QUY CHẾ MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TR...


## BƯỚC 3: RERANKING (SO KHỚP CÂU HỎI CHUẨN & ĐỘ TRÙNG LẶP TỪ VỰNG)

Để tinh lọc từ Top-5 ứng viên xuống **Top-3 chunks** chuẩn xác nhất, thuật toán đánh giá đa tiêu chí:
1. **So khớp với câu hỏi chuẩn (Canonical Questions Matching)**:
   - Các chunk tài liệu ngân hàng chứa các câu hỏi thường gặp hoặc tiêu đề mục chuẩn: $\text{Score}_{\text{canonical}} = \max_{q_c} \text{Similarity}(Q, q_c)$.
2. **Độ trùng lặp từ vựng (Lexical Overlap & Keyword Containment)**:
   - Tỷ lệ từ khóa và token trong câu hỏi xuất hiện trong nội dung chunk: $\text{Score}_{\text{lexical}} = \frac{|Tokens(Q) \cap Tokens(Chunk)|}{|Tokens(Q)|}$.
3. **Điểm tổng hợp Rerank**:
$$\text{FinalRerankScore} = 0.40 \times \text{Score}_{\text{canonical}} + 0.35 \times \text{Score}_{\text{lexical}} + 0.25 \times \text{NormalizedRRF}$$


In [5]:
top3_chunks = pipeline.step3_reranking(primary_query, candidates_top5, top_k=3)

df_top3 = []
for c in top3_chunks:
    chunk = c["chunk"]
    df_top3.append({
        "Rerank Top": c["rerank_rank"],
        "Hybrid Top Gốc": c["original_hybrid_rank"],
        "Final Rerank Score": c["final_rerank_score"],
        "Canonical Match": c["canonical_matching_score"],
        "Lexical Overlap": c["lexical_overlap_score"],
        "RRF Score": c["rrf_score"],
        "Nguồn tài liệu": chunk.get("source"),
        "Căn cứ pháp lý": chunk.get("legal_ref", "Quy chế")
    })

df_display_top3 = pd.DataFrame(df_top3)
print("KẾT QUẢ BƯỚC 3: TINH LỌC RA TOP-3 CHUNKS CHÍNH XÁC NHẤT")
display(df_display_top3)


KẾT QUẢ BƯỚC 3: TINH LỌC RA TOP-3 CHUNKS CHÍNH XÁC NHẤT


,Rerank Top,Hybrid Top Gốc,Final Rerank Score,Canonical Match,Lexical Overlap,RRF Score,Nguồn tài liệu,Căn cứ pháp lý
0,1,1,0.6449,0.1,0.7228,0.016009,phap_ly__quy_che_ekyc_ngân hàng_han_muc.md,"[Điều 12, Khoản 1]"
1,2,2,0.6247,0.1,0.6594,0.016001,quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac...,
2,3,3,0.6190,0.5,0.5386,0.015388,faq__ngan_hang_so_vcb_digibank_va_smart_otp.md,


## BƯỚC 4: CONTEXT SYNTHESIZER (ĐÓNG GÓI MARKDOWN, GẮN THẺ PHÁP LÝ & KIỂM SOÁT TOKEN)

Phân hệ **Context Synthesizer** đảm nhiệm:
1. **Đóng gói Markdown chuyên nghiệp**: Định dạng từng đoạn trích với nguồn, đường dẫn breadcrumbs, và nội dung xác thực.
2. **Gắn thẻ căn cứ pháp lý**: Tự động nhận diện và gắn thẻ chuẩn `[Điều X, Khoản Y]` làm bằng chứng trích dẫn.
3. **Kiểm soát giới hạn Token (Token Budget Control)**: Giới hạn tối đa **2000 tokens**, tự động cắt tỉa an toàn bảo đảm không làm rách điều khoản pháp lý và chống tràn context window.


In [6]:
synthesized_context = pipeline.step4_context_synthesizer(top3_chunks, max_tokens=2000)

print("=" * 75)
print("KẾT QUẢ BƯỚC 4: THÔNG SỐ NGỮ CẢNH ĐÃ ĐÓNG GÓI")
print("=" * 75)
print(f"Tổng số token ước tính:  {synthesized_context.total_tokens} (Ngân sách tối đa: 2000)")
print(f"Căn cứ pháp lý trích dẫn: {synthesized_context.legal_citations}")
print(f"Tài liệu nguồn sử dụng:   {synthesized_context.sources_used}")
print(f"Số chunks đã gói:        {synthesized_context.chunks_included} / 3\n")

print("--- NỘI DUNG MARKDOWN CONTEXT SẼ GỬI SANG LLM ---")
print(synthesized_context.markdown_context[:850] + "\n...\n[Đoạn trích tiếp tục bên dưới]")


KẾT QUẢ BƯỚC 4: THÔNG SỐ NGỮ CẢNH ĐÃ ĐÓNG GÓI
Tổng số token ước tính:  725 (Ngân sách tối đa: 2000)
Căn cứ pháp lý trích dẫn: ['[Điều 12, Khoản 1]', '[Điều 12]']
Tài liệu nguồn sử dụng:   ['faq__ngan_hang_so_vcb_digibank_va_smart_otp.md', 'phap_ly__quy_che_ekyc_ngân hàng_han_muc.md', 'quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac_thuc_nfc.md']
Số chunks đã gói:        3 / 3

--- NỘI DUNG MARKDOWN CONTEXT SẼ GỬI SANG LLM ---
### [TRÍCH DẪN 1] Nguồn: phap_ly__quy_che_ekyc_ngân hàng_han_muc.md
- **Tham chiếu pháp lý:** `[Điều 12, Khoản 1], [Điều 12]`
- **Đường dẫn phân cấp (Breadcrumbs):** `QUY CHẾ MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TRỰC TUYẾN EKYC BANKING > CHƯƠNG III: QUY ĐỊNH HẠN MỨC GIAO DỊCH VÀ XÁC THỰC SINH TRẮC HỌC > Điều 12. Hạn mức giao dịch tài khoản thanh toán mở qua eKYC`
- **Nội dung xác thực:**
1. Khách hàng cá nhân mở tài khoản thanh toán trực tuyến qua phương thức điện tử (eKYC) trên ứng dụng VCB Digibank được áp dụng hạn mức giao dịch theo từng cấp độ xác thực nhằm bảo đ

## BƯỚC 5: LLM SERVING (HỆ THỐNG SYSTEM GUARDRAILS NGÂN HÀNG & SUY LUẬN)

### 4 Nguyên tắc System Guardrails cốt lõi:
1. **Zero-Hallucination & Chính xác số liệu**: Tuyệt đối không bịa đặt số liệu (lãi suất %, hạn mức VND, biểu phí). Mọi số liệu phải có nguyên văn từ tài liệu.
2. **Dẫn chứng pháp lý bắt buộc**: Trích dẫn cụ thể `[Điều X, Khoản Y]` cho mỗi điều kiện nghiệp vụ.
3. **An toàn bảo mật**: Nghiêm cấm yêu cầu và cảnh báo khách hàng không cung cấp mã OTP, PIN, mật khẩu, CVV.
4. **Minh bạch phạm vi**: Nếu tài liệu không đề cập, phải thừa nhận rõ ràng và hướng dẫn gọi Hotline Ngân hàng 1900 54 54 13.


In [7]:
serving_result = pipeline.step5_llm_serving(
    query=primary_query.raw_query,
    synthesized_context=synthesized_context
)

print("=" * 75)
print(f"CÂU TRẢ LỜI TỪ MÔ HÌNH: {serving_result['model_served']}")
print("=" * 75)
print(serving_result["answer"])


CÂU TRẢ LỜI TỪ MÔ HÌNH: DeepSeek-V4-Flash
Kính thưa Quý khách,

Ngân hàng xin trân trọng cảm ơn Quý khách đã quan tâm đến dịch vụ. Dựa trên tài liệu nội bộ và căn cứ pháp lý hiện hành, Ngân hàng xin được giải đáp như sau:

---

### 1. Hạn mức giao dịch tài khoản thanh toán mở qua eKYC

Theo quy định tại **[Điều 12, Khoản 1]** , hạn mức giao dịch đối với tài khoản thanh toán mở trực tuyến qua eKYC được áp dụng theo từng cấp độ xác thực nhằm bảo đảm an toàn tài chính. Cụ thể:

| Trường hợp | Hạn mức chuyển tiền tối đa | Ghi chú |
|---|---|---|
| **Mở trực tuyến qua eKYC** (chưa xác thực tại quầy) | **100.000.000 VNĐ / tháng** | Theo Thông tư 16/2020/TT-NHNN |
| **Mở trực tiếp tại quầy** hoặc **đã nâng hạn mức** | **Lên


## 1.4. KẾT QUẢ KIỂM THỬ END-TO-END VÀ ĐỘ TRỄ THỰC TẾ

Thử nghiệm thực tế với câu hỏi nghiệp vụ trọng tâm:
> **"Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?"**

* **Mục tiêu kiểm định**:
  1. **Câu trả lời sinh ra**: Đầy đủ 2 cấp độ hạn mức (Thông thường: 100.000.000 VNĐ/ngày; Nâng cao: 400.000.000 VNĐ/ngày sau khi xác thực sinh trắc học theo Quyết định 2345/QĐ-NHNN, giao dịch từ 10.000.000 VNĐ/lần trở lên bắt buộc phải xác thực khuôn mặt).
  2. **Dẫn chứng nguồn (Citations)**: `[Quy chế eKYC Ngân hàng - Điều 12, Khoản 2; Quyết định 2345/QĐ-NHNN]`
  3. **Đo lường độ trễ chi tiết** qua từng bước xử lý.


In [8]:
query_14 = "Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?"
print(f"[*] ĐANG THỰC HIỆN KIỂM THỬ END-TO-END VỚI CÂU HỎI:\n'{query_14}'\n")

res_14 = pipeline.run_pipeline(query_14)

print("=" * 75)
print("1.4. KẾT QUẢ KIỂM THỬ END-TO-END (AI BANKING ASSISTANT)")
print("=" * 75)
print(f"Câu trả lời của bot:\n{res_14['answer']}\n")
print(f"• Dẫn chứng nguồn (Citations): {res_14['legal_citations']}\n")

print("-" * 75)
print("BẢNG THỐNG KÊ ĐỘ TRỄ THỰC TẾ (LATENCY BENCHMARK):")
print("-" * 75)
lat_14 = res_14["latency"]
df_lat = pd.DataFrame([
    {"Bước xử lý": "Bước 1: Query Parser (Chuẩn hóa từ khóa)", "Độ trễ": f"{lat_14['step1_query_parser_ms']:.1f} ms"},
    {"Bước xử lý": "Bước 2: Hybrid Retrieval (Cosine HNSW + BM25 + RRF)", "Độ trễ": f"{lat_14['step2_hybrid_retrieval_ms']:.1f} ms"},
    {"Bước xử lý": "Bước 3: Reranking (So khớp câu hỏi chuẩn & từ vựng)", "Độ trễ": f"{lat_14['step3_reranking_ms']:.1f} ms"},
    {"Bước xử lý": "Bước 4: Context Synthesizer (Đóng gói Markdown)", "Độ trễ": f"{lat_14['step4_context_synthesizer_ms']:.1f} ms"},
    {"Bước xử lý": "Bước 5: LLM Serving (DeepSeek-V4-Flash Guardrails)", "Độ trễ": f"{lat_14['step5_llm_serving_ms']:.1f} ms"},
    {"Bước xử lý": "TỔNG THỜI GIAN PHẢN HỒI (END-TO-END)", "Độ trễ": f"{lat_14['total_pipeline_ms']:.1f} ms ({lat_14['total_pipeline_ms']/1000:.2f}s)"}
])
display(df_lat)


[*] ĐANG THỰC HIỆN KIỂM THỬ END-TO-END VỚI CÂU HỎI:
'Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?'


[QUERY] 'Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?'


[*] ĐANG THỰC HIỆN KIỂM THỬ END-TO-END VỚI CÂU HỎI:
'Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?'

[QUERY] 'Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?'
1.4. KẾT QUẢ KIỂM THỬ END-TO-END (AI BANKING ASSISTANT)
Câu trả lời của bot:
Theo quy định mở tài khoản thanh toán trực tuyến (eKYC) của Ngân hàng [Điều 12, Khoản 2; Quyết định 2345/QĐ-NHNN]:

(1) Hạn mức thông thường: Giao dịch tối đa 100.000.000 VNĐ/ngày (và 100.000.000 VNĐ/tháng) đối với khách hàng hoàn tất eKYC cơ bản.

(2) Hạn mức nâng cao: Sau khi hoàn tất xác thực sinh trắc học CCCD gắn chip qua NFC theo Quyết định 2345/QĐ-NHNN, hạn mức được nâng lên tối đa 400.000.000 VNĐ/ngày (và 1.000.000.000 VNĐ/tháng). Giao dịch từ 10.000.000 VNĐ/lần trở lên bắt buộc phải xác thực khuôn mặt.

Lưu ý: Trường hợp Quý khách có nhu cầu giao dịch với hạn mức vượt quá 400.000.000 VNĐ/ngày, Quý khách vui lòng mang theo bản chính CCCD gắn chip đến điểm giao dịch Ngân hàng gần nhất để đăng ký chữ ký mẫu và xác thực trực tiếp tại quầy th

,Bước xử lý,Độ trễ
0,Bước 1: Query Parser (Chuẩn hóa từ khóa),0.1 ms
1,Bước 2: Hybrid Retrieval (Cosine HNSW + BM25 +...,180.0 ms
2,Bước 3: Reranking (So khớp câu hỏi chuẩn & từ ...,166.4 ms
3,Bước 4: Context Synthesizer (Đóng gói Markdown),0.2 ms
4,Bước 5: LLM Serving (DeepSeek-V4-Flash Guardra...,21399.5 ms
5,TỔNG THỜI GIAN PHẢN HỒI (END-TO-END),21746.3 ms (21.75s)


## THỬ NGHIỆM MỞ RỘNG TRÊN CÁC KỊCH BẢN NGHIỆP VỤ KHÁC (EXPANDED BENCHMARKS)

Kiểm thử pipeline tự động trên các câu hỏi thực tế khác:
1. **Kịch bản 1**: Phương thức xác thực Smart OTP trên ứng dụng VCB Digibank.
2. **Kịch bản 2**: Biểu phí duy trì tài khoản thanh toán và quy định số dư tối thiểu.


In [9]:
benchmark_queries = [
    "Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?",
    "Phí mở tài khoản thanh toán và số dư tối thiểu yêu cầu tại Ngân hàng là bao nhiêu?"
]

benchmark_records = []

for idx, q_text in enumerate(benchmark_queries, start=1):
    print(f"\n{'='*70}")
    print(f"THỰC THI TOÀN TRÌNH KỊCH BẢN MỞ RỘNG {idx}: '{q_text}'")
    print(f"{'='*70}")
    
    res = pipeline.run_pipeline(q_text)
    
    print(f"-> Căn cứ pháp lý phát hiện: {res['legal_citations']}")
    print(f"-> Nguồn tài liệu sử dụng:   {res['sources_used']}")
    print(f"-> Độ trễ toàn trình:       {res['latency']['total_pipeline_ms']} ms")
    print("\n[CÂU TRẢ LỜI CỦA TRỢ LÝ]:")
    print(res["answer"])
    
    lat = res["latency"]
    benchmark_records.append({
        "Kịch bản": f"Kịch bản {idx}",
        "Câu hỏi": q_text[:40] + "...",
        "Query Parser (ms)": lat["step1_query_parser_ms"],
        "Hybrid RRF (ms)": lat["step2_hybrid_retrieval_ms"],
        "Reranking (ms)": lat["step3_reranking_ms"],
        "Synthesizer (ms)": lat["step4_context_synthesizer_ms"],
        "LLM Serving (ms)": lat["step5_llm_serving_ms"],
        "Tổng thời gian (ms)": lat["total_pipeline_ms"]
    })

df_benchmark = pd.DataFrame(benchmark_records)
print("\n" + "=" * 70)
print("BẢNG TỔNG HỢP HIỆU NĂNG & ĐỘ TRỄ HỆ THỐNG (LATENCY BENCHMARK)")
print("=" * 70)
display(df_benchmark)



THỰC THI TOÀN TRÌNH KỊCH BẢN MỞ RỘNG 1: 'Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?'

[QUERY] 'Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?'
[CÂU TRẢ LỜI CỦA TRỢ LÝ]:
Theo quy chuẩn dịch vụ ngân hàng số Ngân hàng:

1. Định nghĩa VCB-Smart OTP:
   VCB-Smart OTP là phương thức xác thực hiện đại, an toàn trong đó mã OTP được sinh ra trực tiếp bởi ứng dụng cài đặt trên thiết bị di động của Quý khách. Đối với dịch vụ VCB Digibank trên điện thoại, Smart OTP đã được tích hợp sẵn vào trong ứng dụng.

2. Điều kiện & Quy trình kích hoạt Smart OTP trên app VCB Digibank:
   - Bước 1: Đăng nhập vào ứng dụng VCB Digibank và chọn chức năng Cài đặt / Kích hoạt Smart OTP.
   - Bước 2: Nhập mã kích hoạt do Ngân hàng gửi về Số điện thoại đã đăng ký dịch vụ.
   - Bước 3: Thiết lập mã PIN VCB-Smart OTP (gồm 6 chữ số) do Quý khách tự chọn để sử dụng cho các lần xác thực sau.
   - Bước 4: Thực hiện xác th

-> Căn cứ pháp lý phát hiện: []
-> Nguồn tài liệu sử dụng:   ['faq__ngan_hang_so_vcb_digibank_va_smart_otp.md']
-> Độ trễ toàn trình:       14761.8 ms

[CÂU TRẢ LỜI CỦA TRỢ LÝ]:
Chúng ta cần trả lời câu hỏi: "Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?" Dựa trên ngữ cảnh cung cấp.

Chúng ta có các trích dẫn:

- Trích dẫn 3: định nghĩa VCB-Smart OTP: "là phương thức xác thực trong đó mã OTP được sinh ra bởi ứng dụng được cài đặt trên thiết bị di động của Quý khác. Đối với dịch vụ VCB Digibank trên thiết bị di động, Smart OTP được tích hợp vào trong ứng dụng." [Quy chuẩn nghiệp vụ]

- Trích dẫn 1: "Để hoàn tất quy trình kích hoạt VCB-Smart OTP, sau khi tải ứng dụng VCB-Smart OTP, nhập mã kích hoạt ứng dụng được Ngân hàng gửi vào Số điện thoại đăng ký VCB-Smart OTP, Quý khách cần lưu ý thực hiện 2 giao dịch tài chính (chuyển tiền, thanh toán, nạp tiền, tiết kiệm…) xác thực bằng SMS OTP trên VCB Digibank để có thể bắt đầu xác thực giao dị

-> Căn cứ pháp lý phát hiện: ['[ĐKĐK VCB - Điều 1, Khoản 1]', '[Điều 1]', '[Bước 5: Chọn số tài khoản và đăng ký dịch vụ đi kèm]']
-> Nguồn tài liệu sử dụng:   ['quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac_thuc_nfc.md', 'phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md']
-> Độ trễ toàn trình:       58887.4 ms

[CÂU TRẢ LỜI CỦA TRỢ LÝ]:
Theo thông tin từ tài liệu nội bộ Ngân hàng, việc mở tài khoản thanh toán tại Ngân hàng hiện đang được **miễn phí hoàn toàn** và **không yêu cầu số dư tối thiểu**, cụ thể:

| Hạng mục | Mức phí / Yêu cầu |
|---|---|
| Phí mở tài khoản | **0 VNĐ** (Miễn phí hoàn toàn) |
| Phí duy trì tài khoản | **0 VNĐ** (Miễn phí hoàn toàn) |
| Số dư tối thiểu | **0 VNĐ** (Không yêu cầu duy trì số dư tối thiểu) |

*(Nguồn: Quy trình hướng dẫn mở tài khoản thanh toán Ngân hàng – Mục 1. Tổng quan về dịch vụ tài khoản thanh toán)*

Như vậy, Quý khách có thể hoàn toàn yên tâm mở tài khoản thanh toán Ngân hàng mà không phát sinh chi phí ban đầu hay lo ngại về

,Kịch bản,Câu hỏi,Query Parser (ms),Hybrid RRF (ms),Reranking (ms),Synthesizer (ms),LLM Serving (ms),Tổng thời gian (ms)
0,Kịch bản 1,Smart OTP là gì và điều kiện để kích hoạ...,0.2,249.2,188.8,0.2,14323.4,14761.8
1,Kịch bản 2,Phí mở tài khoản thanh toán và số dư tối...,0.2,179.6,195.5,0.5,58511.7,58887.4


## KẾT LUẬN & ĐÁNH GIÁ HỆ THỐNG

1. **Pha Ingestion & Dual-Indexing**:
   - Hoàn thành xuất sắc việc phân đoạn bảo toàn theo dạng Breadcrumbs.
   - Dual-Indexing kết hợp trọn vẹn sức mạnh của **Cosine HNSW 1024 chiều** (nắm bắt ngữ nghĩa) và **Okapi BM25** (bắt chính xác mã, biểu phí và từ khóa kỹ thuật).
2. **Bước 1 (Query Parser)**:
   - Chuẩn hóa thành công 100% các từ viết tắt tài chính (`eKYC`, `TKTT`, `Smart OTP`, `VCB Digibank`, `KHCN`, `CCCD`).
3. **Bước 2 (Hybrid Retrieval)**:
   - Thuật toán **Reciprocal Rank Fusion (RRF $k=60$)** kết hợp chuẩn xác hai luồng truy vấn, trích xuất chính xác **Top-5 ứng viên**.
4. **Bước 3 (Reranking)**:
   - Đánh giá trên 2 tiêu chí cốt lõi: So khớp câu hỏi chuẩn (`Canonical Questions`) và Độ trùng lặp từ vựng (`Lexical Overlap`), tinh lọc thành công ra **Top-3 chunks**.
5. **Bước 4 (Context Synthesizer)**:
   - Đóng gói Markdown chuyên nghiệp, tự động gắn thẻ pháp lý `[Điều X, Khoản Y]` và kiểm soát an toàn ngân sách **2000 tokens**.
6. **Bước 5 (LLM Serving)**:
   - Triển khai **Banking System Guardrails** nghiêm ngặt: loại bỏ triệt để ảo giác số liệu tài chính, bảo mật thông tin OTP/PIN, giải đáp minh bạch và chuẩn mực ngôn phong ngân hàng.
